# Differential Equations — Session 13
## Section 4.1: Theory of Linear Equations

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. State the existence–uniqueness theorem for an $n$th-order linear IVP.
2. Distinguish initial-value and boundary-value problems.
3. Use linearity and superposition.
4. Test linear independence using the Wronskian.
5. Define a fundamental set and construct a general homogeneous solution.
6. Decompose a nonhomogeneous solution into complementary and particular parts.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–18 min | IVPs, BVPs, existence and uniqueness |\n| 18–35 min | Linear operators and superposition |\n| 35–58 min | Linear independence and Wronskians |\n| 58–75 min | Fundamental sets and general solutions |\n| 75–87 min | Nonhomogeneous decomposition |\n| 87–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 4.1-A — Linear $n$th-order equation

$$
a_n(x)y^{(n)}+a_{n-1}(x)y^{(n-1)}+\cdots+a_0(x)y=g(x).
$$

It is homogeneous when $g(x)=0$.

### Theorem 4.1-B — Existence and uniqueness

If $a_0,\ldots,a_n,g$ are continuous on an interval $I$ and $a_n(x)\ne0$ on $I$, then for every $x_0\in I$ and initial data

$$
y(x_0)=b_0,\quad y'(x_0)=b_1,\ldots,y^{(n-1)}(x_0)=b_{n-1},
$$

there exists exactly one solution on $I$.

### Definition 4.1-C — Boundary-value problem

A BVP prescribes conditions at two or more different points. Unlike an IVP, it may have one, many, or no solutions.

### Theorem 4.1-D — Superposition for homogeneous equations

If $L[y_j]=0$, then $L[c_1y_1+\cdots+c_ky_k]=0$.

### Definition 4.1-E — Wronskian

$$
W(y_1,\ldots,y_n)=\det\left[y_j^{(i-1)}\right]_{i,j=1}^n.
$$

For $n$ solutions of the same homogeneous $n$th-order linear equation, nonzero Wronskian on $I$ is equivalent to linear independence.

### Definition 4.1-F — Fundamental set

A fundamental set is a set of $n$ linearly independent solutions. Its span is the entire homogeneous solution space.

### Theorem 4.1-G — Nonhomogeneous general solution

If $y_p$ is one particular solution and $y_c$ is the general solution of $L[y]=0$, then

$$
y=y_c+y_p
$$

is the general solution of $L[y]=g$.

### Classroom Checkpoint — Why the Wronskian Matters

What does a nonzero Wronskian of $n$ solutions of the same homogeneous $n$th-order linear equation tell us?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. IVP uniqueness versus BVP behavior

For $y''+\omega^2y=0$, the general solution is

$$
y=c_1\cos(\omega x)+c_2\sin(\omega x).
$$

Two initial conditions at the same point determine $(c_1,c_2)$ uniquely. Two endpoint conditions can produce different outcomes.

In [ ]:
def bvp_family(omega=4.0,b=np.pi/2,right_value=0.0):
    # y(0)=0 forces c1=0. The second condition is c2 sin(omega b)=right_value.
    s=np.sin(omega*b)
    x=np.linspace(0,b,500)
    if abs(s)<1e-8 and abs(right_value)<1e-8:
        for c2 in [-2,-1,0.5,1,2]: plt.plot(x,c2*np.sin(omega*x),label=f'c2={c2}')
        status='infinitely many solutions'
    elif abs(s)<1e-8:
        plt.text(.5,.5,'No curve satisfies both conditions',transform=plt.gca().transAxes,ha='center')
        status='no solution'
    else:
        c2=right_value/s; plt.plot(x,c2*np.sin(omega*x),linewidth=3)
        status='one solution'
    plt.scatter([0,b],[0,right_value],s=70); plt.xlabel('x'); plt.ylabel('y'); plt.title(status); plt.legend(); plt.show()
    print('sin(omega b)=',s,'=>',status)
if WIDGETS_AVAILABLE:
    interact(bvp_family,omega=FloatSlider(min=.5,max=8,step=.5,value=4),b=FloatSlider(min=.25,max=3.2,step=.05,value=np.pi/2),right_value=FloatSlider(min=-2,max=2,step=.25,value=0))
else: bvp_family()

## 2. Linearity is an algebraic structure

Let

$$
L[y]=y''-3y'+2y.
$$

Linearity means

$$
L[\alpha f+\beta g]=\alpha L[f]+\beta L[g].
$$

In [ ]:
x=sp.symbols('x',real=True); a,b=sp.symbols('a b')
f=sp.exp(x); g=sp.exp(2*x)
def L(expr): return sp.diff(expr,x,2)-3*sp.diff(expr,x)+2*expr
display(sp.simplify(L(a*f+b*g)-(a*L(f)+b*L(g))))

## 3. Wronskian as a coordinate-volume test

For $y_1=e^x$ and $y_2=e^{2x}$,

$$
W=e^{3x}\ne0.
$$

The functions form a basis for the solution space of $y''-3y'+2y=0$.

In [ ]:
x=sp.symbols('x',real=True)
sets=[[sp.exp(x),sp.exp(2*x)],[sp.sin(x),2*sp.sin(x)],[1,x,x**2]]
for funcs in sets:
    print(funcs); display(wronskian_symbolic(funcs,x))

In [ ]:
def fundamental_family(c1=1.0,c2=1.0):
    x=np.linspace(-2,3,600); y=c1*np.exp(x)+c2*np.exp(2*x)
    plt.plot(x,y,linewidth=2); plt.plot(x,c1*np.exp(x),linestyle='--',label='c1 e^x'); plt.plot(x,c2*np.exp(2*x),linestyle=':',label='c2 e^{2x}')
    plt.title('Coordinates in a two-dimensional solution space'); plt.legend(); plt.show()
if WIDGETS_AVAILABLE: interact(fundamental_family,c1=FloatSlider(min=-3,max=3,step=.25,value=1),c2=FloatSlider(min=-3,max=3,step=.25,value=1))
else: fundamental_family()

## 4. Complementary plus particular response

For

$$
y''-3y'+2y=x,
$$

one particular solution is $y_p=x/2+3/4$. Therefore

$$
y=c_1e^x+c_2e^{2x}+\frac{x}{2}+\frac34.
$$

In [ ]:
x=sp.symbols('x',real=True); yp=x/2+sp.Rational(3,4)
display(sp.simplify(sp.diff(yp,x,2)-3*sp.diff(yp,x)+2*yp-x))

## Classroom Checkpoint — Exit Check

Why does zero initial data force the zero solution for a homogeneous linear IVP?

> Pause here. Let students commit to an answer before running the next cell.